# Piecewise Conversion

Real converters don't have a single efficiency. A condensing boiler is most efficient at mid-load (cool flue gas, full condensation), worse at low load (cycling/standby losses) and worse at high load (hotter flue). A heat pump's COP varies with ambient temperature; a CHP shifts heat-to-power ratio with output.

You'll meet **`ConversionCurve`** — a piecewise-linear coupling between two or more flows. The optimizer picks an operating point on the curve at every timestep.

In [ ]:
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

from fluxopt import Carrier, ConversionCurve, Converter, Effect, Flow, Port, optimize

pio.renderers.default = 'notebook_connected'

## 1. The curve

A 100 kW condensing boiler with three efficiency segments:

| Gas range (kW) | Slope (η = heat/gas) | Why |
|---|---|---|
| 0 – 30  | 0.75 | Cycling and standby losses dominate |
| 30 – 70 | 0.90 | Sweet spot — flue cool enough to condense |
| 70 – 100 | 0.67 | Flue gas hotter, less condensation |

Breakpoints: gas at `[0, 30, 70, 100]` paired with heat at `[0, 22.5, 58.5, 78.5]`.

In [ ]:
gas_bp = [0, 30, 70, 100]
heat_bp = [0, 22.5, 58.5, 78.5]
fig = px.line(x=gas_bp, y=heat_bp, markers=True, height=300, labels={'x': 'gas input (kW)', 'y': 'heat output (kW)'})
fig.update_layout(template='plotly_white', margin={'l': 50, 'r': 20, 't': 10, 'b': 40})
fig

## 2. Solve with `ConversionCurve`

`Converter` takes a `conversion=` argument; the simplest form is a `ConversionCurve` mapping each flow's `short_id` to a list of breakpoints.

We use 24 hours of mixed demand so the optimizer visits every segment. The boiler is built explicitly with `Converter(...)` — `Converter.boiler(...)` is for constant-η boilers; piecewise needs the general form.

In [ ]:
n = 24
timesteps = [datetime(2024, 1, 15) + timedelta(hours=h) for h in range(n)]
demand = np.array(
    [
        10,
        10,
        15,
        20,
        25,
        35,
        50,
        65,
        70,
        60,
        50,
        45,
        40,
        35,
        30,
        25,
        20,
        30,
        55,
        70,
        78,
        75,
        60,
        35,
    ]
)

In [ ]:
result = optimize(
    timesteps=timesteps,
    carriers=[Carrier('gas'), Carrier('heat')],
    effects=[Effect('cost', unit='EUR')],
    ports=[
        Port('gas_grid', imports=[Flow('gas', size=500, effects_per_flow_hour={'cost': 0.05})]),
        Port(
            'workshop', exports=[Flow('heat', size=float(demand.max()), fixed_relative_profile=demand / demand.max())]
        ),
    ],
    converters=[
        Converter(
            'boiler',
            inputs=[Flow('gas', short_id='fuel')],
            outputs=[Flow('heat', size=100)],
            conversion=ConversionCurve({'fuel': gas_bp, 'heat': heat_bp}),
        ),
    ],
    objective_effects='cost',
)
print(f'Total cost: {result.objective:.2f} EUR')

## 3. Read the result

Two views worth looking at:

- **Operating points** — overlay each timestep's `(gas, heat)` on the curve. Their density along each segment shows which efficiency regime the boiler ran in.
- **Instantaneous efficiency** — `heat / gas` over time, dropping in segments where the boiler is forced off the sweet spot.

In [ ]:
gas_rate = result.flow_rate('boiler(fuel)').values
heat_rate = result.flow_rate('boiler(heat)').values

fig = go.Figure()
fig.add_trace(go.Scatter(x=gas_bp, y=heat_bp, mode='lines+markers', name='curve', line_color='#636efa'))
fig.add_trace(
    go.Scatter(
        x=gas_rate,
        y=heat_rate,
        mode='markers',
        name='operating points',
        marker={'symbol': 'x', 'size': 9, 'color': '#ef553b'},
    )
)
fig.update_layout(
    height=350,
    margin={'l': 50, 'r': 20, 't': 10, 'b': 40},
    template='plotly_white',
    xaxis_title='gas input (kW)',
    yaxis_title='heat output (kW)',
)
fig

In [ ]:
times = result.flow_rates.coords['time'].values
eta = np.where(gas_rate > 1e-6, heat_rate / np.where(gas_rate > 1e-6, gas_rate, 1), np.nan)

df = pd.concat(
    [
        pd.DataFrame({'time': times, 'value': heat_rate, 'panel': 'rates (kW)', 'series': 'heat out'}),
        pd.DataFrame({'time': times, 'value': gas_rate, 'panel': 'rates (kW)', 'series': 'gas in'}),
        pd.DataFrame({'time': times, 'value': demand, 'panel': 'rates (kW)', 'series': 'demand'}),
        pd.DataFrame({'time': times, 'value': eta, 'panel': 'efficiency', 'series': 'η'}),
    ],
    ignore_index=True,
)
fig = px.line(df, x='time', y='value', color='series', facet_row='panel', line_shape='hv', height=500)
fig.update_yaxes(matches=None, title='')
fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))
fig.update_layout(template='plotly_white', margin={'l': 50, 'r': 20, 't': 30, 'b': 20})
fig

Look for two things in the operating-points plot: a **cluster on the middle segment** (the η=0.90 sweet spot) where demand is in the 22–58 kW window, and a few points pushed into the **third segment** during the evening peak when demand exceeds 58 kW. The efficiency timeseries drops correspondingly.

`ConversionCurve` is not limited to two flows — pass three (or more) breakpoint vectors of the same length and they are all tied to the same interpolation weights. That's how a CHP with load-dependent heat-to-power ratio is modeled.

The default `method='auto'` lets the underlying `linopy` formulation pick: `lp` for matching-curvature inequalities, `incremental` for monotonic equality breakpoints, otherwise `sos2` (SOS-2 binaries).

## Recap

`ConversionCurve` replaces a fixed efficiency with a piecewise-linear coupling. Pair it with a general `Converter(inputs=..., outputs=..., conversion=...)` (not `Converter.boiler` — that's a constant-η shortcut). For converters that also need to be on/off-able, see the on/off recap in [T3 Status](03-status.ipynb): put the `status` on the conversion curve via `ConversionCurve(..., status=Status(...))` and the binary gates the whole curve at once.

You've now seen the full vocabulary: `Carrier`, `Flow`, `Port`, `Converter`, `Effect`, `Storage`, `Status`, `Sizing`, `Investment`, `ConversionCurve`. Browse the [examples](examples/heat-system.ipynb) for fully-worked scenarios, or jump to the [API reference](../api/) to see what else each object can do.